LSTM PER SENTIMENTS ANALYSIS: OLTRE LA MEMORIA A BREVE TERMINE

Una LST per sentimens Analysis serve a classificare un testo, per esempio una recensione, come positiva o negativa, cercando di mantenere informazioni utili anche quando compaiono molto token prima del punto in cui servono.

Il problema nasce dalle RNN semplici. Una RNN legge una sequenza: token 1 -> token 2 -> token 3 -> ... -> token n
E porta avanti uno stato nascosto.
Il problema è che, quando la sequenza diventa lunga, alcune informazioni importanti possono essere difficili da mentenere.
Durante il training può comparire il vanisching gradient, che rende più difficile imparare dipendenze lontane.

Una LSTM, Long Short-Term Memory, è stata progettata proprio per gestire meglio questo problema.
In una recensione come:
"The movie started very slowly and some scenes were boring, but the ending was absolutely fantastic."
La parola "boring" spinge verso una recensione negativa, ma più avanti abbiamo "but... fantastic" che cambiano molto l'interpretazione.
Il modello deve quindi considerare l'intera sequenza e non fermarsi alla prime parole emotivamente forti.
L'embedding fa ciò che abbiamo già visto (token ID -> vettore denso), la LST riceve quindi una sequenza di vettori, non riceve direttamente le parole.
Il cuore della LSTM è che mantiene due tipi di stato:
    - hidden staste
    - cell state: permette di trasportare l'informazione lungo la sequenza in maniera controllata, conservando il contesto semantico per lunghi periodi. Immaginalo come un nastro traportatore che attraversa l'intera sequenza, la rete può scrivere sul nastro, leggere o cancellare informazioni non più utili.
Ma chi decide cosa resta sul nastro e cosa finisce nel cestino?
Il controllo di questo flusso avviene tramite 3 gate, che possiamo immaginare come a delle valvole.
La LSTM utilizza 3 gate (cancelli) principali:
    - Forget gate: decide quanto dell'informazione precedente mantenere. Questa informazione mi serve ancora? Quali informazioni del passato non sono più rilevanti e quindi da togliere dal nastro. (applica il filtro di dimenticanza)
    - Input gate: decide quale informazione incorporare. Questa nuova parola è importante?. Quali pezzi di 'nuova informazione' aggiungere al nastro delle memoria a lungo termine. (aggiunge nuove informazioni a quelle vecchie che abbiamo già deciso di mantenere)
    - Output gate:  decide quale parte dello stasto corrente utilizzare per produrre il nuovo hidden state. Quale informazione devo rendere disponibile adesso? quale parte della memoria interna deve essere esposta come stato nascosto per il prossimo step?
Quindi una LSTM cerca di mantenere informazioni rilevanti anche attraverso sequenze più lunghe rispetto a una RNN semplice.
La Cell State Update è l'operazione matematica che aggiorna la memoria combinando il vecchio stato con le nuove scoperte.
E' un sistema di pesi e contrappesi che mantiene il sentimet coerente dall'inizio alla fine.
Ogni memento produce sia l'output per le fasi successive sia uno stato nascosto con la memoria che è stato deciso di mentenere, che diventa la nuova memoria a breve termine utilizzata nel passaggio successivo.
A differenza delle RNN classiche, le LSTM non sovrascrivoon l'intera memoria a ogni step, ma effettuano solo aggiornamenti mirati.

Classificatore Binario con LSTM
Costruire la pipelne in TensorFlow
In Keras, implementare una LSTM per il Sentiment Analysis richiede una sequenza precisa di layer: Embedding (per tradurre le parole in vettori), LSTM e un layer di output denso.
La configurazione standard prevede l'uso di 'binary_crossentropy' come funzione di perdita e un'attivazione 'sigmoid' finale per mappare il risultato tra 0 e 1
E' il setup standard per rispondere al sentiment analysis.

Ma quali sono le manopole che possiamo regolare?

Layer e Iperparametri
Configurazione nel modello sequenziale.
- Layer Embedding: trasforma gli indici delle parole (ID) in vettori densi che catturano le relazioni semantiche iniziali.
- Layer LSTM: processa la sequenza di vettori. Il parametro 'units' definisce la complessità della memoria interna, quante sfumature la nostra rete riesce a catturare. Ma non esagerare per non cadere nell'overfitting.
- Global MaXPooling: spesso usato dopo la LSTM per estrarre le feature più saliente dall'intera sequenza temporale. Serve a estrarre il segnale emotivo più forte rilevato durante tutta la frase. Serve per il classificatore finale, semplificandogli il laovro di estrazione.
- Dense Layer: l'ultimo passo che riduce la dimensionalità a un singolo valore di probabilità per la classe positiva.

Le LSTM sono potenti ma tendono a memorizzare il rumore, per questo usiamo il dropout alle connessioni temporali interne.
Monitorare la val_loss è critico: le LSTM tendono a memorizzare il dataset di training se lasciate girare troppo a lungo.

Una volta addestrato, come interpretiamo il vedertto?

Configurazione dell'Output
Interpretazione probabilistica.
L'output finale sarà la nostra sigmoide che andrà a restituirci valori tra 0 e 1
Per la classificazione binaria, l'output y rappresenta la probabiltà che il testo appartenga alla classe positiva (sentiment positivo).
La soglia di decisione standard è fissata a 0.5 per discriminare tra le due classi.

LSTM vs Modelli Classici
Perchè la memoria fa la diffrenza
I modelli classici come Naive Bayes o SVM con TF-IDF considerano il testo come un sacco di parole (Bag of Words), perdendo completamente l'ordine.
Le LSTM, essendo sensibile all'ordine, catturano la struttura sintattica che definisce il reale significato emotivo di un periodo complesso.

Sui casi reali, dove vince la LSTM e dove perde?
Punti di forza e di debolezza
- Dipendenze a lungo raggio: le LSTM eccellono dove il sentiment è definito da parole distanti tra loro. Se il senso di una frase di chiarisce solo dopo 10 parole le reti classiche falliscono, le LSTM no.
- Robustezza semantica: capacità di distinguere sfrumature ironiche o sarcasmo meglio dei modelli statistici
- Costo computazionale: le LSTM sono significativamente più lente da addestrare rispetto a una Logistic Regression. una RNN si allena in millisencodi, una LSTM richiede tempo ed  una GPU
- Data Hunger: richiedono dataset molto più grandi per convergere rispetto ai modelli lineari classici. Se avetete solo poche decine di esempi, un modello classico potrebbe battere una LSTM, perèà è più semplice.

Casi D'Uso reali
- Recensioni lunghe: nei dataset tipo IMDB o articoli di giornale, dove le recensioni superano le 200 parole, la superiorità delle LSTM è schiacciante.
- Messaggi Brevi (es. Twitter o comandi vocali): su testi molto brevi, un modello classico ben ottimizzato può talvolta eguagliare una LSTM con meno risorse.
- Tranfer Learning: quest è un asso nella manca. Se carichi vettori pre-addestrati come GloVe o Word2Vec la LSTM inizia l'addestramento avendo già un'idea di cosa significano le parole del mondo reale, riducendo drasticamente il tempo di convergenza.

Vantagio Competitivo
Perchè l'ordine cambia tutto
L'ordine conta ed in statistica la probabiltà congiunta delle parole non è commutativa.
Un modello BoW non distingue tra 'film non brutto' d a'film brutto non'. La LSTM si, grazie alla conservazione dell'ordine temporale.
Questa capacità di 'allineamento temporale' permette di mappare concetti astratti in un modo impossibile per le semplici conta di frequenza.
Non stiamo solo guardando una foto del testo 

In [1]:
"""
============================================================
ARCHITETTURA DI UNA RETE LSTM IN KERAS (BACKEND PYTORCH)
============================================================

Focus: Input -> Embedding -> LSTM -> Output.
"""

import os
# Impostiamo il backend a PyTorch (Best Practice Keras 3)
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers

def create_lstm_network(vocab_size=10000, max_len=200):
    """
    Definisce la struttura di una rete LSTM per Sentiment Analysis.
    
    Questa funzione utilizza la Functional API di Keras 
    per creare un grafo computazionale dove il testo fluisce 
    attraverso la memoria della LSTM.
    """
    
    # 1. DEFINIZIONE DELL'INPUT
    # Rappresenta una sequenza di numeri (indici di parole) di lunghezza fissa.
    inputs = keras.Input(shape=(max_len,), name="input_sequenza")

    # 2. LAYER EMBEDDING
    # Trasforma ogni numero in un vettore denso (es. 128 dimensioni).
    # È qui che il modello impara il "significato" iniziale delle parole.
    embedding = layers.Embedding(
        input_dim=vocab_size, 
        output_dim=128, 
        name="livello_embedding"
    )(inputs)

    # 3. LAYER LSTM (Il nucleo della lezione)
    # Parametri chiave:
    # - units=64: Dimensione dello stato nascosto e della 'Cell State'.
    # - dropout=0.2: Protezione contro l'overfitting (spegne neuroni a caso).
    # - return_sequences=False: Restituisce solo l'ultimo stato (il "riassunto" della frase).
    lstm_layer = layers.LSTM(
        units=64, 
        dropout=0.2, 
        recurrent_dropout=0.2, 
        name="memoria_lstm"
    )(embedding)

    # 4. LIVELLO DI OUTPUT
    # Un singolo neurone con attivazione Sigmoide.
    # Trasforma la memoria della LSTM in una probabilità (0=Negativo, 1=Positivo).
    outputs = layers.Dense(1, activation="sigmoid", name="classificatore")(lstm_layer)

    # 5. CREAZIONE DEL MODELLO
    # Specifichiamo dove inizia e dove finisce il flusso dei dati.
    model = keras.Model(inputs=inputs, outputs=outputs, name="Rete_LSTM_Semplice")

    # COMPILAZIONE
    # Configuriamo come il modello deve imparare.
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    
    return model

# --- VISUALIZZAZIONE ---

if __name__ == "__main__":
    # Creiamo l'istanza del modello
    my_lstm = create_lstm_network()

    # Mostriamo lo schema dell'architettura a video
    # Perfetto per mostrare agli studenti come cambiano le dimensioni dei dati (Shape)
    my_lstm.summary()

"""
============================================================
SPIEGAZIONE DETTAGLIATA DEL CODICE
============================================================
1. INPUT: Riceve la frase già trasformata in numeri (batch_size, 200).
2. EMBEDDING: Aggiunge una dimensione vettoriale (batch_size, 200, 128).
3. LSTM: È qui che avviene il "Gating". Il layer elabora i 200 step temporali 
   uno dopo l'altro, ma grazie alla Cell State non dimentica l'inizio.
   L'output viene ridotto a (batch_size, 64).
4. DENSE: Prende i 64 concetti estratti dalla LSTM e decide il sentiment finale.
============================================================
"""

Model: "Rete_LSTM_Semplice"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_sequenza (InputLayer)     │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ livello_embedding (Embedding)   │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ memoria_lstm (LSTM)             │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classificatore (Dense)          │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

'\n============================================================\nSPIEGAZIONE DETTAGLIATA DEL CODICE\n============================================================\n1. INPUT: Riceve la frase già trasformata in numeri (batch_size, 200).\n2. EMBEDDING: Aggiunge una dimensione vettoriale (batch_size, 200, 128).\n3. LSTM: È qui che avviene il "Gating". Il layer elabora i 200 step temporali \n   uno dopo l\'altro, ma grazie alla Cell State non dimentica l\'inizio.\n   L\'output viene ridotto a (batch_size, 64).\n4. DENSE: Prende i 64 concetti estratti dalla LSTM e decide il sentiment finale.\n============================================================\n'